In [0]:
# Databricks notebook source
# DBTITLE 1,Initialize Windowing Configurations
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import sys

source_table = "workspace.silver.grocery_prices"
target_table = "workspace.gold.grocery_prices"
export_volume_path = "/Volumes/workspace/gold/gold_output/"

try:
    print(f"Reading from Silver Layer: {source_table}")
    silver_df = spark.read.table(source_table)

    # DBTITLE 2,Filter for Core 10 Basket Items (Broad Matching Support)
    # Using specific keyword signatures present in StatCan table strings
    target_basket = [
        "Milk", "Eggs", "Bread", "Butter", "Chicken", 
        "Bananas", "Potatoes", "Beef", "Coffee", "Bacon"
    ]
    
    # Construct a flexible regex string to isolate our target products
    regex_filter = "(?i)" + "|".join(target_basket)
    
    basket_df = silver_df.filter(F.col("ProductName").rlike(regex_filter))

    # DBTITLE 3,Calculate Analytical Window (Month-over-Month Price Shifting)
    # Window partitions data by item + region and orders chronologically
    product_window = (Window
                      .partitionBy("Geography", "ProductName")
                      .orderBy("SnapshotDate"))

    gold_df = (basket_df
               # Retrieve previous month's value
               .withColumn("PreviousMonthPrice", F.lag("AveragePrice", 1).over(product_window))
               # Calculate percentage difference formula: ((Current - Prev) / Prev) * 100
               .withColumn("MoM_PercentageChange", 
                           F.round(
                               ((F.col("AveragePrice") - F.col("PreviousMonthPrice")) / F.col("PreviousMonthPrice")) * 100, 
                               2
                           ))
               .select(
                   "SnapshotDate",
                   "Geography",
                   "ProductName",
                   "AveragePrice",
                   F.coalesce(F.col("MoM_PercentageChange"), F.lit(0.0)).alias("MoM_PercentageChange")
               ))

    # DBTITLE 4,Commit To Gold Unity Catalog Mart
    (gold_df.write
     .format("delta")
     .mode("overwrite")
     .saveAsTable(target_table))
    print(f"Gold Analytical Table saved to {target_table}.")

    # DBTITLE 5,Consolidate Output for Tableau Public
    # Make sure target folder directory exists
    dbutils.fs.mkdirs(export_volume_path)
    
    # Coalesce pushes all data into 1 file partition block to prevent messy multi-file fragments
    (gold_df.coalesce(1).write
     .format("csv")
     .option("header", "true")
     .mode("overwrite")
     .save(export_volume_path + "toronto_grocery_index_extract"))
     
    print(f"Tableau Public Flat File dropped cleanly to volume: {export_volume_path}")

except Exception as e:
    print(f"ANALYTICAL PROCESSING ERROR in Gold Pipeline: {str(e)}")
    sys.exit(1)